# A deep agent, assembled

A deep agent can work through a longer task while planning, using tools, asking specialists, and staying inside controls.

There is no new magic part. We will combine parts we already built:

- the tool loop;
- a policy sub-agent;
- an approval gate;
- a budget;
- context measurement; and
- verification.

Before the final framework section, install the framework extras once with: pip install -e ".[langgraph]". The hand-built section uses the normal course environment.

In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import litellm
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent
load_dotenv(course_root / ".env")
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
litellm.drop_params = True
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=str(course_root / ".litellm_cache"))

## 1. Choose a small queue

We use two tickets so each run is easy to follow.

In [ ]:
from shoplab import world

tickets = {
    item["ticket_id"]: item
    for rows in world.load_tickets().values()
    for item in rows
}
orders = {item["order_id"]: item for item in world.load_orders()}
customers = {item["customer_id"]: item for item in world.load_customers()}

QUEUE = ["TKT-2224", "TKT-2225"]

for ticket_id in QUEUE:
    ticket = tickets[ticket_id]
    print(ticket_id, ticket["reason_text"][:70])
    print("known:", ticket["gold"])

## 2. Add a policy sub-agent

The parent gives the worker one policy question. The worker gets only the search tool and returns one short finding.

In [ ]:
from shoplab.context import run_subagent
from shoplab.tools import Tool, standard_tools

RESEARCH_SYSTEM = (
    "Search policy once. Return one sentence with the policy id and rule. "
    "Do not decide the ticket."
)

def make_policy_tool():
    def research_policy(question):
        result = run_subagent(
            question,
            {"search_policy": standard_tools()["search_policy"]},
            model=MODEL,
            system=RESEARCH_SYSTEM,
            max_steps=4,
        )
        return {"finding": result["answer"]}

    return Tool(
        "research_policy",
        "Ask a policy worker one focused question.",
        {
            "type": "object",
            "properties": {"question": {"type": "string"}},
            "required": ["question"],
        },
        research_policy,
    )

print(make_policy_tool().fn("What rule applies to an opened item returned in time?"))

## 3. Approve side effects with shop rules

The model may request a refund. The gate checks the exact order and amount against the deterministic rule result.

In [ ]:
from shoplab.rules import decide

def make_approver(ticket):
    correct = decide(
        ticket,
        orders[ticket["order_id"]],
        customers[ticket["customer_id"]],
    )

    def approve(tool_name, arguments):
        if arguments.get("order_id") != ticket["order_id"]:
            return False

        if tool_name == "issue_refund":
            expected = correct["refund_usd"]
            return (
                correct["decision"] in {"approve_refund", "partial_refund"}
                and expected is not None
                and abs(arguments.get("amount_usd", 0) - expected) <= 0.01
            )

        if tool_name == "create_replacement":
            return (
                correct["decision"] == "replacement"
                and arguments.get("sku") == ticket["sku"]
            )

        return True

    return approve

demo_ticket = tickets["TKT-2224"]
demo_gate = make_approver(demo_ticket)
print("wrong amount:", demo_gate("issue_refund", {"order_id": demo_ticket["order_id"], "amount_usd": 1}))
print("right amount:", demo_gate("issue_refund", {"order_id": demo_ticket["order_id"], "amount_usd": 36.50}))

## 4. Assemble the agent

The function below creates fresh tools, budget, and ledger for each ticket. Fresh state keeps one ticket from affecting another.

In [ ]:
import shoplab.llm
from shoplab.controls import Budget, require_approval
from shoplab.loop import run_agent
from shoplab.tools import Ledger

SYSTEM = (
    "Handle the return. Look up the order and customer. Ask research_policy once. "
    "Use calc for arithmetic. Carry out an approved refund, replacement, or escalation. "
    "Then call finish with decision, policy_id, and refund_usd."
)

def clear_ticket(ticket):
    ledger = Ledger()
    budget = Budget(max_calls=16, max_cost_usd=0.05)
    base = standard_tools(ledger)

    allowed = ("get_order", "get_customer", "calc", "escalate", "finish")
    tools = {name: base[name] for name in allowed}
    tools["research_policy"] = make_policy_tool()

    approver = make_approver(ticket)
    for name in ("issue_refund", "create_replacement"):
        tools[name] = require_approval(base[name], approver)

    task = json.dumps({key: value for key, value in ticket.items() if key != "gold"})

    def charge(step, message):
        budget.charge(shoplab.llm.LEDGER[-1]["cost_usd"])

    result = run_agent(
        task,
        tools,
        model=MODEL,
        system=SYSTEM,
        max_steps=14,
        on_step=charge,
    )

    return {
        "ticket": ticket,
        "answer": result.answer,
        "messages": result.messages,
        "actions": ledger.entries,
        "budget": budget.snapshot(),
        "stop_reason": result.stop_reason,
    }

In [ ]:
runs = [clear_ticket(tickets[ticket_id]) for ticket_id in QUEUE]

for run in runs:
    print(run["ticket"]["ticket_id"])
    print("answer:", run["answer"])
    print("actions:", run["actions"])
    print("budget:", run["budget"])

## 5. Grade and verify the queue

Evaluation tells us whether the returned fields match. Verification independently recomputes the shop decision.

In [ ]:
from shoplab.evals import score_ticket
from shoplab.verify import check_decision

for run in runs:
    ticket = run["ticket"]
    answer = run["answer"] if isinstance(run["answer"], dict) else {}
    score = score_ticket(answer, ticket["gold"])
    check = check_decision(
        answer,
        ticket,
        orders[ticket["order_id"]],
        customers[ticket["customer_id"]],
    )
    print(ticket["ticket_id"], "exact:", score["exact"], "verified:", check["agrees"])

The full policy text stayed inside the research worker. The parent received only its finding. This is the context boundary from chapter 10.

## 6. The same idea in deepagents

The deepagents library packages planning, files, and sub-agents as maintained middleware. The next cells build a read-only version of the same brief.

In [ ]:
framework_agent = None

try:
    from deepagents import create_deep_agent
    from langchain_openai import ChatOpenAI

    provider_model = MODEL.split("/", 1)[1] if MODEL.startswith("openrouter/") else MODEL
    chat_model = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
        model=provider_model,
        temperature=0,
    )

    def get_order(order_id: str) -> dict:
        """Look up one Larkspur order."""
        return orders.get(order_id, {"error": "order not found"})

    def get_customer(customer_id: str) -> dict:
        """Look up one Larkspur customer."""
        return customers.get(customer_id, {"error": "customer not found"})

    def find_policy(query: str) -> list:
        """Search Larkspur policy documents."""
        return world.search_policy(query)

    framework_agent = create_deep_agent(
        model=chat_model,
        tools=[get_order, get_customer, find_policy],
        system_prompt="Look up the facts and return a short Larkspur ticket decision.",
    )
    print(type(framework_agent))
except ImportError as error:
    print("deepagents is not installed:", error)

In [ ]:
if framework_agent is not None:
    framework_result = framework_agent.invoke({
        "messages": [{"role": "user", "content": json.dumps(tickets["TKT-2205"])}]
    })
    print(framework_result["messages"][-1].content)
else:
    print("framework example skipped")

The framework gives us an assembled, maintained agent. Our hand-built version makes every budget, gate, and context boundary visible. The right choice depends on whether convenience or control matters more.

## What to remember

1. A deep agent is a composition of familiar parts.
2. Sub-agents can protect the parent context.
3. Deterministic rules can approve exact side effects.
4. Budgets keep long runs bounded.
5. Evaluation and verification check the result.
6. Frameworks package the same machinery behind stronger defaults.

## Small practice steps

1. Add one more ticket to QUEUE.
2. Lower the call budget and observe where the run stops.
3. Remove the policy worker and compare the parent context.